# AgroMind v4 (final): yield-grounded crop ranking + conformal prediction sets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravixalgorithm/agromind/blob/main/notebooks/AgroMind_v4_Novel_Method.ipynb)

**Research question.** Published crop recommenders report ~99% accuracy, mostly on the Kaggle crop table split at random.
How good are they on **real Government of India data**, in **states and years the model has never seen**, and can we do better?

* **Data:** DES district × season × year crop statistics (1997-98 to 2022-23, official DES portal) + NASA POWER climate + SoilGrids soil (built by `backend/scripts/build_large_dataset.py`).
* **Task:** for each district-season-year, rank the crops grown there by realised yield percentile. The "best crop" is the top-yielding one.
* **Baselines (as in the literature):** Gaussian NB, Random Forest, XGBoost, MLP classifiers, plus naive *crop prior* and *last-year persistence*.
* **Ours (YieldRank v4, final):** a blend, per district-season, of a listwise ranker (XGBoost rank:ndcg) and a pointwise yield-percentile regressor (LightGBM) on (district-season, crop) pairs, plus 0.25 × last-year persistence. The configuration was chosen on held-out training states and years, never on test folds (`experiments/select_final.py`). Recommendations come with **split-conformal prediction sets** at a 90% coverage target.
* **Protocols:** random 5-fold, **leave-states-out** (5 folds over states), **leave-years-out** (5 contiguous year blocks).

Everything below is computed live; nothing is pasted in. The code lives in `experiments/run.py` (fetched from GitHub on Colab).

In [1]:
# Setup: works locally (repo checkout) and on Colab (downloads from GitHub)
import os, sys, subprocess, urllib.request
from pathlib import Path
RAW = "https://raw.githubusercontent.com/ravixalgorithm/agromind/main/"
root = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "experiments/run.py").exists()), None)
if root is None:  # Colab
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm", "xgboost", "pyarrow"], check=True)
    root = Path("agromind");
    for f in ["experiments/run.py", "backend/data/large/agromind_india_large.parquet", "backend/data/Crop_recommendation.csv"]:
        (root / f).parent.mkdir(parents=True, exist_ok=True)
        if not (root / f).exists():
            urllib.request.urlretrieve(RAW + f, root / f)
sys.path.insert(0, str(root / "experiments"))
import numpy as np, pandas as pd, run
from scipy.stats import wilcoxon
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("repo root:", root.resolve())

repo root: /Users/originkit/.config/baton/worktrees/agromind-334992/claude-5


## 0. How AgroMind got here: v1 → v2 → v3 → v4 (read from `docs/research/results.json`)

In [2]:
import json
R = json.load(open(root / "docs/research/results.json")) if (root / "docs/research/results.json").exists() else {}
g = lambda *ks, d=None: (lambda v: v)(__import__("functools").reduce(lambda a, k: a.get(k, {}) if isinstance(a, dict) else {}, ks, R) or d)
v1 = g("v1", "RandomForest", "reported_accuracy"); v1x = g("v1", "GaussianNB", "exact_match_accuracy")
v2 = g("models", "RandomForest (v2 deployed)", "accuracy_mean"); v2l = max([v.get("accuracy", 0) for k, v in g("loso", d={}).items() if "closed" in k] or [0])
v3 = g("v3", "leave_states_out", "ExtraTrees: climate + season + lat/lon", "hit1_mean"); v3b = g("v3", "leave_states_out", "baseline: season popularity", "hit1_mean")
n4 = g("final", "numbers", d=None) or g("v4", "numbers", d={})
print(f"v1  Kaggle table, 2,200 rows, random 80/20  -> RF accuracy {100*(v1 or 0):.1f}%.  Weakness: near-synthetic data, no location/year; "
      f"its multi-output decoder hides errors (exact-match GNB {100*(v1x or 0):.1f}%).")
print(f"v2  67,200 merged rows, 74 crops, random 5-fold -> RF accuracy {100*(v2 or 0):.1f}%.  Weakness: train on two sources, test on the third "
      f"-> at most {100*v2l:.1f}% (leave-one-source-out).")
print(f"v3  400,379 Govt APY rows, grouped splits -> unseen states hit@1 {100*(v3 or 0):.1f}% (ExtraTrees; season-popularity baseline {100*(v3b or 0):.1f}%; "
      "v3 hit@1 counts any of the top-3 yield crops).  Weakness: no uncertainty, classifier framing.")
print(f"v4  same Govt data, best-yield crop per district-season-year -> unseen states hit@1 {n4.get('OursHitLSO','?')}% vs best literature classifier "
      f"{n4.get('BaseHitLSO','?')}% and persistence {n4.get('PersHitLSO','?')}%; served 90% sets (state-robust) cover {n4.get('CCSRCovLSO','?')}% with "
      f"{n4.get('CCSRSizeLSO','?')} crops, worst state {n4.get('SRWorstCov','?')}%.  Remaining weakness: per-state coverage, single country (see paper).")

v1  Kaggle table, 2,200 rows, random 80/20  -> RF accuracy 98.0%.  Weakness: near-synthetic data, no location/year; its multi-output decoder hides errors (exact-match GNB 86.9%).
v2  67,200 merged rows, 74 crops, random 5-fold -> RF accuracy 94.3%.  Weakness: train on two sources, test on the third -> at most 32.6% (leave-one-source-out).
v3  400,379 Govt APY rows, grouped splits -> unseen states hit@1 41.8% (ExtraTrees; season-popularity baseline 30.4%; v3 hit@1 counts any of the top-3 yield crops).  Weakness: no uncertainty, classifier framing.
v4  same Govt data, best-yield crop per district-season-year -> unseen states hit@1 56.3% vs best literature classifier 38.3% and persistence 54.4%; served 90% sets (state-robust) cover 94.2% with 3.66 crops, worst state 84.5%.  Remaining weakness: per-state coverage, single country (see paper).


## 1. The literature's setting: Kaggle crop table, random 80/20 split
This reproduces the ~99% figures (2,200 rows, 100 per crop, no location or year).

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
k = pd.read_csv(root / "backend/data/Crop_recommendation.csv")
Xtr, Xte, ytr, yte = train_test_split(k.drop(columns="label"), k.label, test_size=0.2, stratify=k.label, random_state=0)
for name, m in [("Random Forest", RandomForestClassifier(300, random_state=0)), ("Gaussian NB", GaussianNB())]:
    print(f"{name:14s} accuracy on Kaggle random split: {100*(m.fit(Xtr, ytr).predict(Xte) == yte).mean():.2f}%")

Random Forest  accuracy on Kaggle random split: 99.55%
Gaussian NB    accuracy on Kaggle random split: 99.55%


## 2. Real data: DES district-season-year crop statistics

In [4]:
df = run.load()
crops = sorted(df.crop_slug.unique())
QX, Qy, Q = run.query_table(df, crops, hist=True)
print(f"(query, crop) rows : {len(df):,}")
print(f"queries            : {df.qid.nunique():,}  (district x season x year)")
print(f"crops / states     : {len(crops)} / {df.state.nunique()}   years {df.year.min()}-{df.year.max()}")
print("candidates per query:", df.groupby('qid').size().describe()[['mean','50%','max']].round(1).to_dict())
print("soil source share  :", df.soil_source.value_counts(normalize=True, dropna=False).round(3).to_dict())
df[['state','district','year','season','crop_slug','yield','yield_pct','temperature','rainfall','lag1_pct']].head(8)

(query, crop) rows : 259,181
queries            : 43,443  (district x season x year)
crops / states     : 49 / 36   years 1997-2022
candidates per query: {'mean': 6.0, '50%': 5.0, 'max': 27.0}
soil source share  : {'soilgrids_centroid': 0.998, 'state_median': 0.002, nan: 0.0}


,state,district,year,season,crop_slug,yield,yield_pct,temperature,rainfall,lag1_pct
0,Andaman and Nicobar Islands,Nicobars,2000,Kharif,arecanut,1.5949,0.7639,27.54,907.4,NaN
1,Andaman and Nicobar Islands,Nicobars,2000,Kharif,rice,3.1471,0.9107,27.54,907.4,NaN
2,Andaman and Nicobar Islands,Nicobars,2010,Rabi,arecanut,1.7055,0.6901,27.23,1256.3,NaN
3,Andaman and Nicobar Islands,Nicobars,2010,Rabi,cashewnut,0.2604,0.2619,27.23,1256.3,NaN
4,Andaman and Nicobar Islands,Nicobars,2011,Rabi,arecanut,1.1390,0.5063,27.67,886.9,0.6901
5,Andaman and Nicobar Islands,Nicobars,2011,Rabi,cashewnut,0.2750,0.4754,27.67,886.9,0.2619
6,Andaman and Nicobar Islands,Nicobars,2012,Rabi,arecanut,1.4200,0.6479,27.92,926.8,0.5063
7,Andaman and Nicobar Islands,Nicobars,2012,Rabi,cashewnut,0.2901,0.3500,27.92,926.8,0.4754


## 3. The method (source shown for transparency)
`fit_rank` = YieldRank (LambdaMART on yield-percentile relevance); `conformal` = LAC / APS split-conformal sets.

In [5]:
import inspect
print(inspect.getsource(run.fit_rank)); print(inspect.getsource(run.conformal))

def fit_rank(df, tr, crops, hist, objective, target="yield", **override):
    """objective: lambdarank | regression | xendcg | xgb_rank. target: yield (yield_pct) or share
    (current area share, i.e. 'what farmers grow most': the no-yield-target ablation)."""
    d = df.loc[tr]
    X = pair_features(d, crops, hist)
    y = d.yield_pct if target == "yield" else d.share.groupby(d.qid).rank(pct=True)
    rel = np.clip(np.floor(y * 10), 0, 9).astype(int)  # graded 0..9
    group = d.groupby("qid", sort=False).size().values
    params = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1, n_jobs=NJ)
    gain = override.pop("label_gain", list(range(10)))
    trunc = override.pop("lambdarank_truncation_level", 10)
    params |= override
    return_model = override.pop("return_model", False)
    params |= override
    if objective in ("lambdarank", "xendcg"):
   

In [6]:
MODELS = ["Persistence", "GaussianNB [6,12]", "RandomForest [8,21]", "XGBoost [11,22]", "MLP [13]", "YieldRank (ours)"]
rng = np.random.default_rng(run.SEED)
idx = lambda qs: df.index[df.qid.isin(qs)]

def evaluate(protocol):
    rows, perq, conf = [], [], []
    for f, (trq, teq, groups) in enumerate(run.folds(df, protocol)):
        a, b = next(run.GroupShuffleSplit(1, test_size=0.2, random_state=run.SEED + f).split(trq, groups=groups.loc[trq].values))
        grp_cal, rest = trq[b], trq[a]
        iid_cal = rng.choice(rest, size=len(grp_cal), replace=False)
        fit_i, te_i, gc_i, ic_i = idx(np.setdiff1d(rest, iid_cal)), idx(teq), idx(grp_cal), idx(iid_cal)
        for name in MODELS:
            score = run.MODELS[name](df, fit_i, crops, True, QX, Qy)
            s_te = score(te_i)
            pq = run.per_query(df.loc[te_i], s_te)
            s_gc, s_ic = score(gc_i), score(ic_i)
            T = run.fit_T(df.loc[ic_i, "qid"], s_ic, df.loc[ic_i, "best"].values)
            mk = lambda i, s: df.loc[i, ["qid", "best"]].assign(p=run.softmax_by_q(df.loc[i, "qid"], s, T))
            ci, per_i = run.conformal(mk(ic_i, s_ic), mk(te_i, s_te))
            cg, per_g = run.conformal(mk(gc_i, s_gc), mk(te_i, s_te))
            r = dict(fold=f, model=name.split(" [")[0], ndcg3=pq["ndcg@3"].mean(), hit1=100*pq["hit@1"].mean(), hit3=100*pq["hit@3"].mean(),
                     cov_iid=100*ci["LAC@0.9"]["coverage"], size_iid=ci["LAC@0.9"]["size"],
                     cov_heldout=100*cg["LAC@0.9"]["coverage"], size_heldout=cg["LAC@0.9"]["size"])
            rows.append(r)
            print(f"fold {f} | {r['model']:13s} | NDCG@3 {r['ndcg3']:.3f} | Hit@1 {r['hit1']:5.1f}% | Hit@3 {r['hit3']:5.1f}% | "
                  f"90% sets: iid-cal {r['cov_iid']:5.1f}% (size {r['size_iid']:.2f})  held-out-cal {r['cov_heldout']:5.1f}% (size {r['size_heldout']:.2f})", flush=True)
            perq.append(pq.assign(model=r["model"], fold=f).reset_index())
            conf.append(per_i.add_suffix("_iid").join(per_g.add_suffix("_grp")).assign(model=r["model"], fold=f,
                        state=lambda d: Q.state.reindex(d.index).values).reset_index(names="qid"))
    R = pd.DataFrame(rows)
    summ = R.groupby("model", sort=False)[["ndcg3", "hit1", "hit3", "cov_iid", "size_iid", "cov_heldout", "size_heldout"]].agg(["mean", "std"]).round(3)
    return R, summ, pd.concat(perq), pd.concat(conf)

def significance(P, base):
    o = P[P.model == "YieldRank (ours)"].set_index(["fold", "qid"])
    for b in base:
        x = P[P.model == b].set_index(["fold", "qid"]).reindex(o.index)
        for k in ["hit@1", "ndcg@3"]:
            d = (o[k] - x[k]).dropna()
            print(f"YieldRank vs {b:13s} {k:7s}: mean diff {d.mean():+.4f}   Wilcoxon p = {wilcoxon(d).pvalue:.2e}   (n = {len(d):,} test queries)")

## 4. Random split (optimistic, for reference)

In [7]:
R_rand, S_rand, P_rand, C_rand = evaluate("random")
S_rand

fold 0 | Persistence   | NDCG@3 0.915 | Hit@1  55.8% | Hit@3  87.4% | 90% sets: iid-cal  90.9% (size 3.18)  held-out-cal  90.3% (size 3.08)


fold 0 | GaussianNB    | NDCG@3 0.819 | Hit@1  32.5% | Hit@3  73.4% | 90% sets: iid-cal  90.2% (size 4.37)  held-out-cal  90.4% (size 4.39)


fold 0 | RandomForest  | NDCG@3 0.914 | Hit@1  57.6% | Hit@3  88.7% | 90% sets: iid-cal  90.3% (size 2.79)  held-out-cal  89.8% (size 2.74)


fold 0 | XGBoost       | NDCG@3 0.913 | Hit@1  58.1% | Hit@3  88.1% | 90% sets: iid-cal  90.8% (size 2.96)  held-out-cal  90.3% (size 2.89)


fold 0 | MLP           | NDCG@3 0.899 | Hit@1  53.9% | Hit@3  85.6% | 90% sets: iid-cal  90.1% (size 3.17)  held-out-cal  90.1% (size 3.18)


fold 0 | YieldRank (ours) | NDCG@3 0.935 | Hit@1  60.5% | Hit@3  90.7% | 90% sets: iid-cal  90.3% (size 2.72)  held-out-cal  89.8% (size 2.65)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.912 | Hit@1  58.0% | Hit@3  87.5% | 90% sets: iid-cal  89.8% (size 2.95)  held-out-cal  89.5% (size 2.90)


fold 4 | MLP           | NDCG@3 0.898 | Hit@1  53.8% | Hit@3  85.5% | 90% sets: iid-cal  89.8% (size 3.13)  held-out-cal  90.7% (size 3.25)


fold 4 | YieldRank (ours) | NDCG@3 0.935 | Hit@1  60.0% | Hit@3  90.1% | 90% sets: iid-cal  90.1% (size 2.72)  held-out-cal  90.6% (size 2.80)


ndcg3           hit1           hit3        cov_iid        size_iid        cov_heldout        size_heldout       
                   mean    std    mean    std    mean    std    mean    std     mean    std        mean    std         mean    std
model                                                                                                                             
Persistence       0.914  0.001  55.286  0.386  87.098  0.310  90.019  0.633    3.106  0.064      89.840  0.516        3.075  0.057
GaussianNB        0.819  0.002  33.117  0.503  73.018  0.389  90.035  0.490    4.351  0.046      90.392  0.340        4.390  0.044
RandomForest      0.915  0.001  57.528  0.555  88.603  0.182  89.968  0.447    2.772  0.039      90.157  0.374        2.795  0.056
XGBoost           0.912  0.001  57.450  0.699  87.731  0.301  90.038  0.517    2.918  0.043      89.968  0.314        2.908  0.016
MLP               0.898  0.002  53.696  0.593  85.429  0.163  89.962  0.253    3.176  0.048      90.097  0.539        3.191  0.050
YieldRank (ours)  0.935  0.000  60.021  0.372  90.185  0.372  90.215  0.212    2.739  0.028      89.863  0.566        2.697  0.061

## 5. Leave-states-out: train on some states, test on states never seen

In [8]:
R_lso, S_lso, P_lso, C_lso = evaluate("leave-states-out")
S_lso

fold 0 | Persistence   | NDCG@3 0.900 | Hit@1  50.5% | Hit@3  83.6% | 90% sets: iid-cal  88.4% (size 3.34)  held-out-cal  88.7% (size 3.39)


fold 0 | GaussianNB    | NDCG@3 0.745 | Hit@1  25.4% | Hit@3  57.6% | 90% sets: iid-cal  81.5% (size 4.57)  held-out-cal  87.5% (size 5.16)


fold 0 | RandomForest  | NDCG@3 0.815 | Hit@1  32.3% | Hit@3  67.7% | 90% sets: iid-cal  70.2% (size 3.41)  held-out-cal  85.3% (size 5.21)


fold 0 | XGBoost       | NDCG@3 0.823 | Hit@1  34.0% | Hit@3  67.7% | 90% sets: iid-cal  76.0% (size 3.71)  held-out-cal  87.8% (size 5.12)


fold 0 | MLP           | NDCG@3 0.806 | Hit@1  34.3% | Hit@3  68.3% | 90% sets: iid-cal  73.0% (size 3.42)  held-out-cal  87.6% (size 5.02)


fold 0 | YieldRank (ours) | NDCG@3 0.910 | Hit@1  52.4% | Hit@3  85.9% | 90% sets: iid-cal  85.6% (size 2.78)  held-out-cal  88.7% (size 3.20)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.822 | Hit@1  33.0% | Hit@3  71.3% | 90% sets: iid-cal  78.7% (size 3.61)  held-out-cal  89.1% (size 4.69)


fold 4 | MLP           | NDCG@3 0.826 | Hit@1  38.5% | Hit@3  73.3% | 90% sets: iid-cal  73.4% (size 3.11)  held-out-cal  82.5% (size 4.04)


fold 4 | YieldRank (ours) | NDCG@3 0.932 | Hit@1  58.8% | Hit@3  89.9% | 90% sets: iid-cal  88.6% (size 2.64)  held-out-cal  90.7% (size 2.93)


ndcg3           hit1           hit3        cov_iid        size_iid        cov_heldout        size_heldout       
                   mean    std    mean    std    mean    std    mean    std     mean    std        mean    std         mean    std
model                                                                                                                             
Persistence       0.910  0.021  54.427  4.459  86.457  3.396  88.953  3.295    3.049  0.267      89.757  3.443        3.167  0.196
GaussianNB        0.781  0.051  27.815  3.984  66.022  8.487  87.615  9.425    4.367  0.491      90.707  7.097        4.682  0.392
RandomForest      0.825  0.041  34.531  3.618  71.747  7.149  74.230  6.948    3.297  0.255      90.805  3.917        5.112  0.805
XGBoost           0.838  0.038  36.603  4.228  73.547  6.479  79.031  5.242    3.454  0.311      89.920  3.052        4.771  0.598
MLP               0.830  0.045  38.325  5.628  73.400  6.620  75.079  7.869    3.075  0.365      87.567  4.710        4.371  0.611
YieldRank (ours)  0.922  0.020  56.259  4.559  88.433  3.193  87.742  2.869    2.689  0.118      90.308  2.157        3.025  0.213

In [9]:
significance(P_lso, ["Persistence", "GaussianNB", "RandomForest", "XGBoost", "MLP"])

YieldRank vs Persistence   hit@1  : mean diff +0.0183   Wilcoxon p = 8.67e-20   (n = 43,443 test queries)
YieldRank vs Persistence   ndcg@3 : mean diff +0.0128   Wilcoxon p = 2.96e-194   (n = 43,443 test queries)
YieldRank vs GaussianNB    hit@1  : mean diff +0.2844   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs GaussianNB    ndcg@3 : mean diff +0.1415   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs RandomForest  hit@1  : mean diff +0.2173   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)


YieldRank vs RandomForest  ndcg@3 : mean diff +0.0972   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs XGBoost       hit@1  : mean diff +0.1966   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs XGBoost       ndcg@3 : mean diff +0.0846   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs MLP           hit@1  : mean diff +0.1793   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs MLP           ndcg@3 : mean diff +0.0921   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)


### Conformal coverage per held-out state (YieldRank, 90% target, LAC)
`cov_iid` = calibrated on random queries from training states; `cov_heldout` = calibrated on held-out training states.

In [10]:
c = C_lso[C_lso.model == "YieldRank (ours)"]
per_state = (c.groupby("state").agg(test_queries=("qid", "size"), cov_iid=("cov_LAC_iid", "mean"), cov_heldout=("cov_LAC_grp", "mean"),
                                     avg_set_size=("size_LAC_grp", "mean")))
per_state[["cov_iid", "cov_heldout"]] *= 100
per_state = per_state[per_state.test_queries >= 30].sort_values("test_queries", ascending=False).round(2)
print(f"states below 85% coverage: iid-cal {(per_state.cov_iid < 85).sum()} / {len(per_state)},  held-out-cal {(per_state.cov_heldout < 85).sum()} / {len(per_state)}")
per_state

states below 85% coverage: iid-cal 5 / 32,  held-out-cal 2 / 32


,test_queries,cov_iid,cov_heldout,avg_set_size
state,,,,
Uttar Pradesh,5349,86.82,89.96,3.12
Bihar,3596,91.91,93.83,2.78
Madhya Pradesh,3562,86.55,88.35,3.23
Karnataka,2729,86.11,89.74,3.33
Gujarat,2507,85.88,88.23,2.94
Rajasthan,2405,85.49,88.07,3.05
Maharashtra,2354,82.07,86.36,3.13
Odisha,2250,87.16,89.69,2.80
West Bengal,2060,90.19,92.48,2.88


## 6. Leave-years-out: train on some years, test on unseen year blocks

In [11]:
R_lyo, S_lyo, P_lyo, C_lyo = evaluate("leave-years-out")
S_lyo

fold 0 | Persistence   | NDCG@3 0.926 | Hit@1  59.8% | Hit@3  89.2% | 90% sets: iid-cal  92.4% (size 3.18)  held-out-cal  91.5% (size 3.01)


fold 0 | GaussianNB    | NDCG@3 0.812 | Hit@1  31.1% | Hit@3  71.6% | 90% sets: iid-cal  90.6% (size 4.45)  held-out-cal  91.1% (size 4.51)


fold 0 | RandomForest  | NDCG@3 0.903 | Hit@1  53.6% | Hit@3  86.2% | 90% sets: iid-cal  89.7% (size 3.11)  held-out-cal  90.2% (size 3.19)


fold 0 | XGBoost       | NDCG@3 0.905 | Hit@1  54.0% | Hit@3  86.4% | 90% sets: iid-cal  90.1% (size 3.17)  held-out-cal  91.0% (size 3.30)


fold 0 | MLP           | NDCG@3 0.884 | Hit@1  51.1% | Hit@3  84.1% | 90% sets: iid-cal  89.3% (size 3.32)  held-out-cal  90.0% (size 3.43)


fold 0 | YieldRank (ours) | NDCG@3 0.939 | Hit@1  61.3% | Hit@3  91.2% | 90% sets: iid-cal  92.3% (size 2.88)  held-out-cal  90.5% (size 2.64)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.897 | Hit@1  50.8% | Hit@3  83.5% | 90% sets: iid-cal  86.6% (size 3.12)  held-out-cal  91.1% (size 3.85)


fold 4 | MLP           | NDCG@3 0.897 | Hit@1  50.4% | Hit@3  83.6% | 90% sets: iid-cal  87.9% (size 3.17)  held-out-cal  91.6% (size 3.72)


fold 4 | YieldRank (ours) | NDCG@3 0.939 | Hit@1  59.5% | Hit@3  90.5% | 90% sets: iid-cal  90.3% (size 2.72)  held-out-cal  92.9% (size 3.14)


ndcg3           hit1           hit3        cov_iid        size_iid        cov_heldout        size_heldout       
                   mean    std    mean    std    mean    std    mean    std     mean    std        mean    std         mean    std
model                                                                                                                             
Persistence       0.914  0.013  55.143  3.476  87.007  1.930  89.888  1.839    3.105  0.049      89.680  1.727        3.065  0.225
GaussianNB        0.813  0.003  31.661  0.511  71.702  0.158  89.681  0.675    4.372  0.080      89.829  0.959        4.387  0.104
RandomForest      0.899  0.012  52.036  2.181  84.681  1.942  86.222  3.074    2.892  0.133      89.816  2.146        3.360  0.361
XGBoost           0.897  0.014  52.037  2.711  84.214  2.230  86.788  2.525    3.031  0.103      89.469  1.509        3.398  0.312
MLP               0.886  0.012  50.250  1.841  83.148  1.684  87.497  1.992    3.167  0.110      89.986  1.039        3.492  0.175
YieldRank (ours)  0.930  0.014  58.787  2.812  89.528  1.806  89.155  3.139    2.721  0.121      89.701  2.323        2.784  0.298

In [12]:
significance(P_lyo, ["Persistence", "GaussianNB", "RandomForest", "XGBoost", "MLP"])

YieldRank vs Persistence   hit@1  : mean diff +0.0361   Wilcoxon p = 8.68e-72   (n = 43,443 test queries)
YieldRank vs Persistence   ndcg@3 : mean diff +0.0167   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs GaussianNB    hit@1  : mean diff +0.2709   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs GaussianNB    ndcg@3 : mean diff +0.1174   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs RandomForest  hit@1  : mean diff +0.0672   Wilcoxon p = 2.41e-176   (n = 43,443 test queries)
YieldRank vs RandomForest  ndcg@3 : mean diff +0.0315   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs XGBoost       hit@1  : mean diff +0.0673   Wilcoxon p = 9.10e-180   (n = 43,443 test queries)
YieldRank vs XGBoost       ndcg@3 : mean diff +0.0331   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs MLP           hit@1  : mean diff +0.0853   Wilcoxon p = 2.82e-280   (n = 43,443 test queries)
YieldRank vs MLP           ndcg@3 : mean di

## 7. Summary: Hit@1 (%) and NDCG@3 by protocol

In [13]:
summary = pd.concat({"random": S_rand, "leave-states-out": S_lso, "leave-years-out": S_lyo}, axis=0)
summary[[("hit1", "mean"), ("hit1", "std"), ("ndcg3", "mean"), ("cov_iid", "mean"), ("size_iid", "mean"), ("cov_heldout", "mean"), ("size_heldout", "mean")]]

hit1         ndcg3 cov_iid size_iid cov_heldout size_heldout
                                     mean    std   mean    mean     mean        mean         mean
                 model                                                                           
random           Persistence       55.286  0.386  0.914  90.019    3.106      89.840        3.075
                 GaussianNB        33.117  0.503  0.819  90.035    4.351      90.392        4.390
                 RandomForest      57.528  0.555  0.915  89.968    2.772      90.157        2.795
                 XGBoost           57.450  0.699  0.912  90.038    2.918      89.968        2.908
                 MLP               53.696  0.593  0.898  89.962    3.176      90.097        3.191
                 YieldRank (ours)  60.021  0.372  0.935  90.215    2.739      89.863        2.697
leave-states-out Persistence       54.427  4.459  0.910  88.953    3.049      89.757        3.167
                 GaussianNB        27.815  3.984  0.781  87.615    4.367      90.707        4.682
                 RandomForest      34.531  3.618  0.825  74.230    3.297      90.805        5.112
                 XGBoost           36.603  4.228  0.838  79.031    3.454      89.920        4.771
                 MLP               38.325  5.628  0.830  75.079    3.075      87.567        4.371
                 YieldRank (ours)  56.259  4.559  0.922  87.742    2.689      90.308        3.025
leave-years-out  Persistence       55.143  3.476  0.914  89.888    3.105      89.680        3.065
                 GaussianNB        31.661  0.511  0.813  89.681    4.372      89.829        4.387
                 RandomForest      52.036  2.181  0.899  86.222    2.892      89.816        3.360
                 XGBoost           52.037  2.711  0.897  86.788    3.031      89.469        3.398
                 MLP               50.250  1.841  0.886  87.497    3.167      89.986        3.492
                 YieldRank (ours)  58.787  2.812  0.930  89.155    2.721      89.701        2.784

## 8. Check: the paper's numbers, recomputed here
Each value below is computed from this notebook's own runs and compared with `docs/research/results.json` → `v4.numbers`, which is the same file the paper's `numbers.tex` comes from.

In [14]:
import json
paper = json.load(open(root / "docs/research/results.json"))["v4"]["numbers"] if (root / "docs/research/results.json").exists() else {}
ours, ab = "YieldRank (ours)", {"random": "R", "leave-states-out": "LSO", "leave-years-out": "LYO"}
check = {}
for p, (S_, R_) in {"random": (S_rand, R_rand), "leave-states-out": (S_lso, R_lso), "leave-years-out": (S_lyo, R_lyo)}.items():
    m = R_.groupby("model").mean(numeric_only=True)
    base = m.drop(index=["YieldRank (ours)", "Persistence"]).hit1.idxmax()
    check |= {f"OursHit{ab[p]}": f"{m.loc[ours, 'hit1']:.1f}", f"OursNDCG{ab[p]}": f"{m.loc[ours, 'ndcg3']:.3f}",
              f"PersHit{ab[p]}": f"{m.loc['Persistence', 'hit1']:.1f}", f"BaseHit{ab[p]}": f"{m.loc[base, 'hit1']:.1f}", f"BaseName{ab[p]}": base.replace("RandomForest", "Random Forest").replace("GaussianNB", "Gaussian NB"),
              f"CovIID{ab[p]}": f"{m.loc[ours, 'cov_iid']:.1f}", f"SizeIID{ab[p]}": f"{m.loc[ours, 'size_iid']:.2f}",
              f"CovGRP{ab[p]}": f"{m.loc[ours, 'cov_heldout']:.1f}", f"SizeGRP{ab[p]}": f"{m.loc[ours, 'size_heldout']:.2f}"}
for k, v in check.items():
    print(f"{k:14s} notebook {v:>14s}   paper {paper.get(k, '?'):>14s}   {'OK' if paper.get(k) == v else 'DIFF'}")
print("all match:", all(paper.get(k) == v for k, v in check.items()))

OursHitR       notebook           60.0   paper           60.0   OK
OursNDCGR      notebook          0.935   paper          0.935   OK
PersHitR       notebook           55.3   paper           55.3   OK
BaseHitR       notebook           57.5   paper           57.5   OK
BaseNameR      notebook  Random Forest   paper  Random Forest   OK
CovIIDR        notebook           90.2   paper           90.2   OK
SizeIIDR       notebook           2.74   paper           2.74   OK
CovGRPR        notebook           89.9   paper           89.9   OK
SizeGRPR       notebook           2.70   paper           2.70   OK
OursHitLSO     notebook           56.3   paper           56.3   OK
OursNDCGLSO    notebook          0.922   paper          0.922   OK
PersHitLSO     notebook           54.4   paper           54.4   OK
BaseHitLSO     notebook           38.3   paper           38.3   OK
BaseNameLSO    notebook            MLP   paper            MLP   OK
CovIIDLSO      notebook           87.7   paper           87.7 

## Reading the numbers
* The ~99% of Section 1 is a property of the Kaggle table, not of crop choice. On real data, **no** model's top-1 pick matches the best-yielding crop anywhere near that often.
* Compare each baseline's Hit@1 between the random and leave-states-out tables: the drop is the over-optimism of random splits.
* *Persistence* (repeat last year's yield ranking) is a strong, rarely reported baseline. The gap between it and YieldRank is the honest size of the learned gain.
* Conformal coverage with i.i.d. calibration is a guarantee about *seen* states. Per-state coverage shows how it behaves on unseen ones.

The paper (`paper/main.pdf`) reports the same experiment from `experiments/run.py`.